상황 : eda와 모델링을 모두 했지만 몇몇 계열이나 값들이 실종
기본으로 잡았던 파일의 잘못됨을 확인
발표 당일 병합 다시 시도 
eda 다시 시작 전에 코드 날아감 (반성 : 앞으로 깃헙에 바로바로 올릴 것)
급하게 작성된 현 eda 코드

eda 분석 결과와 해석
EDA 결과 해석 - 대학 중도탈락률 영향요인 분석
 1. 타겟 변수 특성
평균 5.42%, 중앙값 4.65% → 우편향 분포 (일부 대학의 극히 높은 탈락률)
최대값 100% → 극단적 케이스 존재할 것임 (폐과/정원미달 등)
이상치 6.4% → 정상범위를 벗어난 대학들이 상당수
 2. 설립구분별 격차 (핵심 발견)
설립구분중도탈락률해석사립5.78%가장 높음 - 재정압박, 선발기준 완화국립4.43%중간 - 안정적 운영, 적정 선발국립대법인2.49%가장 낮음 - 우수한 학생, 충분한 지원
 핵심 인사이트:

사립대의 구조적 문제 (등록금 의존 → 입학기준 완화 → 부적응 학생 증가)
국립대법인의 우수성 (선별적 입학 + 체계적 관리)

 3. 지역별 격차 (수도권 vs 비수도권)
비수도권: 5.99% vs 수도권: 4.47% (1.52%p 차이)
지역별 위험도 순위:

고위험: 제주(10.5%), 전남(7.5%), 경북(7.44%)
중위험: 광주, 전북, 충북 (6-7%)
저위험: 서울(3.93%), 울산(3.17%), 대구(3.03%)

 해석:

수도권 집중효과: 우수 학생, 취업기회, 교육인프라 집중
지방 소멸 위기: 인구감소 → 학령인구 부족 → 입학기준 하락

 4. 상관관계 분석 (예측 모델 힌트)
강한 음의 상관관계 (탈락률 ↓):

재학생충원율 (-0.392): 충원율 높음 = 매력적 대학 = 탈락률 낮음
신입생경쟁률 (-0.374): 경쟁률 높음 = 우수한 학생 = 탈락률 낮음
신입생충원율 (-0.320): 정원 대비 입학생 비율과 반비례

 모델링 시사점:

충원 관련 변수들이 가장 강력한 예측변수
지역/설립구분 카테고리 변수 중요
등록금/장학금은 상관관계 상대적으로 약함

 5. 데이터 품질 이슈
결측치 패턴:

신입생 관련 (8.9%): 일부 대학의 신입생 미모집
취업률 관련 (11.7%): 졸업생 부족한 신설과/특수과
재학생충원 관련 (14.4%): 계산 불가능한 특수 상황

 처리 전략:

결측치는 각 그룹별 중앙값으로 대체
이상치는 99% Percentile Capping 적용

6. 예측 모델 방향성
예상 중요 피처 순위:

재학생충원율, 신입생경쟁률 (수치형)
설립구분, 지역 (범주형)
수도권여부 (이진형)
취업률, 진로진출률 (수치형)

모델 성능 예상:

상관관계 0.3-0.4 수준으로 중간 정도 예측력 기대
앙상블 모델로 비선형 관계 포착 필요

In [ ]:
# =============================================================================
# 대학 특성에 따른 중도탈락률 예측 - 탐색적 데이터 분석 (EDA)
# =============================================================================

# 필요한 라이브러리 설치 및 임포트
!pip install -q seaborn matplotlib plotly pandas numpy scipy

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
from scipy import stats
from scipy.stats import chi2_contingency
import matplotlib.font_manager as fm

warnings.filterwarnings('ignore')

# 한글폰트 설정
import platform
if platform.system() == 'Linux':
    import subprocess
    try:
        subprocess.run(['apt-get', '-qq', 'install', 'fonts-nanum'],
                      stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
    except:
        pass
    import matplotlib.font_manager as fm
    try:
        fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
        plt.rc('font', family='NanumGothic')
    except:
        plt.rc('font', family='DejaVu Sans')
else: 
  # Windows/Mac의 경우 기본 설정
    plt.rc('font', family='DejaVu Sans')

plt.rcParams['axes.unicode_minus'] = False  # 마이너스 기호 깨짐 방지

# # 스타일 설정
# plt.style.use('default')
# sns.set_palette("husl")

print("="*80)
print(" 대학 특성에 따른 중도탈락률 예측 모델 - 탐색적 데이터 분석")
print("="*80)

# =============================================================================
# 1. 데이터 로드 및 기본 정보
# =============================================================================

print("\n 1. 데이터 로드 및 기본 정보")
print("-" * 50)

# 데이터 로드
df = pd.read_csv('final.csv', encoding='utf-8') # 병합 완료된 최종전처리(final)을 뜻함 알아서 수정 

print(f"데이터 형태: {df.shape}")
print(f"컬럼 수: {df.shape[1]}")
print(f"행 수: {df.shape[0]:,}")

print(f"\n 연도별 데이터 분포:")
year_counts = df['기준년도'].value_counts().sort_index()
for year, count in year_counts.items():
    print(f"  {year}년: {count:,}개 ({count/len(df)*100:.1f}%)")

# 기본 정보 출력
print("\n 컬럼 정보:")
print(df.info())

print("\n기술통계:")
print(df.describe())

# =============================================================================
# 2. 결측치 분석
# =============================================================================

print("\n2. 결측치 분석")
print("-" * 50)

# 결측치 확인
missing_data = df.isnull().sum()
missing_percentage = (missing_data / len(df)) * 100
missing_df = pd.DataFrame({
    '결측치 개수': missing_data,
    '결측치 비율(%)': missing_percentage
}).sort_values('결측치 개수', ascending=False)

print("결측치 현황:")
if missing_df['결측치 개수'].sum() == 0:
    print("✅ 결측치가 없습니다!")
else:
    print(missing_df[missing_df['결측치 개수'] > 0])

# =============================================================================
# 3. 타겟 변수 분석 (중도탈락률)
# =============================================================================

print("\n 3. 타겟 변수 분석 - 중도탈락률")
print("-" * 50)

target_col = '중도탈락률(%)'

print(f"중도탈락률 기술통계:")
print(df[target_col].describe())

# 타겟 변수 분포 시각화
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 히스토그램
axes[0,0].hist(df[target_col], bins=50, alpha=0.7, color='skyblue', edgecolor='black')
axes[0,0].axvline(df[target_col].mean(), color='red', linestyle='--', 
                  label=f'평균: {df[target_col].mean():.2f}%')
axes[0,0].axvline(df[target_col].median(), color='orange', linestyle='--', 
                  label=f'중앙값: {df[target_col].median():.2f}%')
axes[0,0].set_title('중도탈락률 분포')
axes[0,0].set_xlabel('중도탈락률 (%)')
axes[0,0].set_ylabel('빈도')
axes[0,0].legend()

# 박스플롯
axes[0,1].boxplot(df[target_col])
axes[0,1].set_title('중도탈락률 박스플롯')
axes[0,1].set_ylabel('중도탈락률 (%)')

# Q-Q plot (정규성 검정)
stats.probplot(df[target_col], dist="norm", plot=axes[1,0])
axes[1,0].set_title('Q-Q Plot (정규성 검정)')

# 연도별 중도탈락률 추이
yearly_dropout = df.groupby('기준년도')[target_col].mean()
axes[1,1].plot(yearly_dropout.index, yearly_dropout.values, marker='o', linewidth=2)
axes[1,1].set_title('연도별 중도탈락률 추이')
axes[1,1].set_xlabel('기준년도')
axes[1,1].set_ylabel('평균 중도탈락률 (%)')
axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 이상치 분석
Q1 = df[target_col].quantile(0.25)
Q3 = df[target_col].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[(df[target_col] < lower_bound) | (df[target_col] > upper_bound)]
print(f"\n이상치 분석:")
print(f"이상치 개수: {len(outliers):,}개 ({len(outliers)/len(df)*100:.1f}%)")
print(f"이상치 기준: < {lower_bound:.2f}% 또는 > {upper_bound:.2f}%")

# =============================================================================
# 4. 범주형 변수 분석
# =============================================================================

print("\n 4. 범주형 변수 분석")
print("-" * 50)

categorical_vars = ['설립구분', '지역', '수도권여부', '대계열']

# 범주형 변수별 분포 확인
for var in categorical_vars:
    if var in df.columns:
        print(f"\n{var} 분포:")
        value_counts = df[var].value_counts()
        for idx, count in value_counts.items():
            print(f"  {idx}: {count:,}개 ({count/len(df)*100:.1f}%)")

# 범주형 변수별 중도탈락률 분석
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for i, var in enumerate(categorical_vars):
    if var in df.columns:
        # 박스플롯
        sns.boxplot(data=df, x=var, y=target_col, ax=axes[i])
        axes[i].set_title(f'{var}별 중도탈락률 분포')
        axes[i].tick_params(axis='x', rotation=45)
        
        # 평균값 표시
        means = df.groupby(var)[target_col].mean().sort_values(ascending=False)
        for j, (category, mean_val) in enumerate(means.items()):
            axes[i].text(j, mean_val + 0.3, f'{mean_val:.1f}%', 
                        ha='center', fontweight='bold', color='red')

plt.tight_layout()
plt.show()

# 통계적 유의성 검정 (ANOVA)
print("\n 범주형 변수별 통계적 유의성 검정 (ANOVA):")
for var in categorical_vars:
    if var in df.columns:
        groups = [group[target_col].values for name, group in df.groupby(var)]
        f_stat, p_value = stats.f_oneway(*groups)
        
        print(f"{var}: F={f_stat:.4f}, p={p_value:.6f} {'***' if p_value < 0.001 else '**' if p_value < 0.01 else '*' if p_value < 0.05 else ''}")

# =============================================================================
# 5. 설립구분별 상세 분석
# =============================================================================

print("\n 5. 설립구분별 상세 분석")
print("-" * 50)

# 설립구분별 주요 지표 비교
key_metrics = ['등록금', '1인당장학금', '취업률(%)', '신입생_경쟁률', 
               '신입생_충원율(%)', '재학생충원율', target_col]

# 존재하는 컬럼만 필터링
available_metrics = [col for col in key_metrics if col in df.columns]

establishment_summary = df.groupby('설립구분')[available_metrics].agg(['mean', 'std']).round(2)
print("설립구분별 주요 지표 요약:")
print(establishment_summary)

# 설립구분별 지표 시각화
n_metrics = len(available_metrics) - 1  # 중도탈락률 제외
n_cols = 3
n_rows = (n_metrics + n_cols - 1) // n_cols

if n_metrics > 0:
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 6*n_rows))
    if n_rows == 1:
        axes = axes.reshape(1, -1)
    axes = axes.flatten()

    for i, metric in enumerate(available_metrics[:-1]):  # 중도탈락률 제외
        sns.boxplot(data=df, x='설립구분', y=metric, ax=axes[i])
        axes[i].set_title(f'설립구분별 {metric}')
        axes[i].tick_params(axis='x', rotation=45)

    # 빈 subplot 숨기기
    for j in range(i+1, len(axes)):
        axes[j].set_visible(False)

    plt.tight_layout()
    plt.show()

# =============================================================================
# 6. 지역별 분석
# =============================================================================

print("\n 6. 지역별 분석")
print("-" * 50)

# 수도권 vs 비수도권 비교
if '수도권여부' in df.columns:
    metro_comparison = df.groupby('수도권여부')[target_col].agg(['mean', 'std', 'count']).round(2)
    print("수도권 vs 비수도권 중도탈락률 비교:")
    print(metro_comparison)

# 지역별 중도탈락률 순위
if '지역' in df.columns:
    regional_dropout = df.groupby('지역').agg({
        target_col: ['mean', 'count']
    }).round(2)
    regional_dropout.columns = ['평균_중도탈락률', '데이터_개수']
    regional_dropout = regional_dropout.sort_values('평균_중도탈락률', ascending=False)
    
    print(f"\n지역별 중도탈락률 순위:")
    print(regional_dropout)

    # 지역별 시각화 (상위 15개 지역만)
    top_regions = regional_dropout.head(15)
    
    plt.figure(figsize=(14, 8))
    bars = plt.bar(range(len(top_regions)), top_regions['평균_중도탈락률'], 
                   alpha=0.7, color='steelblue')
    
    plt.xticks(range(len(top_regions)), top_regions.index, rotation=45)
    plt.ylabel('평균 중도탈락률 (%)')
    plt.title('지역별 평균 중도탈락률 (상위 15개 지역)')
    plt.grid(axis='y', alpha=0.3)

    # 전국 평균선
    national_avg = df[target_col].mean()
    plt.axhline(y=national_avg, color='red', linestyle='--', alpha=0.7, 
                label=f'전국 평균: {national_avg:.1f}%')
    plt.legend()

    # 막대 위에 수치 표시
    for i, (bar, val) in enumerate(zip(bars, top_regions['평균_중도탈락률'])):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                f'{val:.1f}%', ha='center', va='bottom', fontsize=9)

    plt.tight_layout()
    plt.show()

# =============================================================================
# 7. 수치형 변수 분석
# =============================================================================

print("\n 7. 수치형 변수 분석")
print("-" * 50)

# 수치형 변수 선택
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [col for col in numeric_cols if col != '기준년도']  # 연도 제외

print(f"분석할 수치형 변수 개수: {len(numeric_cols)}")

# 주요 수치형 변수들과 타겟 변수 간 상관관계
main_numeric_cols = [col for col in ['신입생_충원율(%)', '신입생_경쟁률', '취업률(%)', '진로진출률(%)', 
                     '1인당장학금', '등록금', '재학생충원율', target_col] if col in df.columns]

if len(main_numeric_cols) > 1:
    correlation_matrix = df[main_numeric_cols].corr()

    # 상관관계 히트맵
    plt.figure(figsize=(12, 10))
    mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))
    sns.heatmap(correlation_matrix, annot=True, cmap='RdBu_r', center=0,
                square=True, mask=mask, fmt='.3f', cbar_kws={"shrink": .8})
    plt.title('주요 변수 간 상관관계 히트맵')
    plt.tight_layout()
    plt.show()

    # 타겟 변수와의 상관관계 순위
    target_correlations = df[numeric_cols].corrwith(df[target_col]).sort_values(key=abs, ascending=False)
    print(f"\n중도탈락률과의 상관관계:")
    print("강한 상관관계 (상위 10개):")
    print(target_correlations.head(10).round(3))

# =============================================================================
# 8. 분포 및 이상치 분석  
# =============================================================================

print("\n 8. 주요 변수 분포 및 이상치 분석")
print("-" * 50)

# 주요 변수들의 분포 시각화
key_vars = [col for col in [target_col, '신입생_경쟁률', '등록금', '1인당장학금', '취업률(%)'] 
            if col in df.columns]

if len(key_vars) > 0:
    n_vars = len(key_vars)
    n_cols = 3
    n_rows = (n_vars + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5*n_rows))
    if n_rows == 1:
        axes = axes.reshape(1, -1)
    axes = axes.flatten()

    for i, var in enumerate(key_vars):
        # 히스토그램과 박스플롯
        axes[i].hist(df[var].dropna(), bins=30, alpha=0.7, edgecolor='black')
        axes[i].set_title(f'{var} 분포')
        axes[i].set_xlabel(var)
        axes[i].set_ylabel('빈도')
        
        # 기본 통계 정보 추가
        mean_val = df[var].mean()
        median_val = df[var].median()
        axes[i].axvline(mean_val, color='red', linestyle='--', alpha=0.7, label=f'평균: {mean_val:.1f}')
        axes[i].axvline(median_val, color='orange', linestyle='--', alpha=0.7, label=f'중앙값: {median_val:.1f}')
        axes[i].legend()

    # 빈 subplot 숨기기
    for j in range(i+1, len(axes)):
        axes[j].set_visible(False)

    plt.tight_layout()
    plt.show()

# 이상치 요약 정보
print(f"\n주요 변수별 이상치 현황 (IQR 방식):")
for var in key_vars:
    Q1 = df[var].quantile(0.25)
    Q3 = df[var].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = df[(df[var] < lower_bound) | (df[var] > upper_bound)]
    print(f"  {var}: {len(outliers):,}개 ({len(outliers)/len(df)*100:.1f}%)")

print("\n" + "="*80)
print(" 탐색적 데이터 분석 완료!")
print(" 다음 단계: 데이터 전처리 및 피처 엔지니어링")
print("="*80)